# Módulo 6 · Sistemas multiagente y CrewMaster

**Duración sugerida:** 5 horas · **Entrega:** workflow multiagente con roles
documentados, intercambio de resultados, registro y evaluación.

## Objetivos del laboratorio

- Saber **cuándo** conviene usar varios agentes.
- Diseñar **roles** (recopilar, analizar, revisar, sintetizar).
- Coordinar la **delegación** y el **intercambio de resultados**.
- Implementar **revisión crítica** con repetición condicional de etapas.
- Introducir **CrewMaster** (operaciones, canales de colaboración, evaluación).

## Conexión con el proyecto ARAS

El equipo multiagente de ARAS divide el trabajo: un agente **busca**, otro
**analiza**, otro **revisa** la consistencia y otro **sintetiza** el informe.

## 1. ¿Cuándo usar varios agentes?

Usa **un agente** cuando la tarea es simple y secuencial. Usa **varios** cuando:

- Hay **etapas diferenciadas** con responsabilidades distintas.
- Quieres **revisión crítica independiente** (un agente no corrige su propio sesgo).
- Necesitas **escalar** (cada rol se especializa y se prueba por separado).

## 2. Dependencias

In [ ]:
%pip install -q langchain-core langchain-ollama langgraph pydantic

## 3. Diseño de roles

Definimos cuatro roles. Cada uno es un nodo con su propio prompt. La coordinación
se expresa como un **grafo**: la revisión puede devolver el flujo a la recopilación
si detecta vacíos (bucle), o continuar a la síntesis.

In [ ]:
from typing import TypedDict, Literal
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langgraph.graph import StateGraph, START, END

llm = ChatOllama(model="llama3.2", temperature=0)

# Colección local que simula las fuentes del investigador.
FUENTES = [
    "La evaluación de agentes mide relevancia, uso correcto de herramientas y trazabilidad.",
    "RAG reduce las alucinaciones al fundamentar la respuesta en documentos recuperados.",
    "MCP estandariza el acceso a herramientas y recursos externos.",
]

class Estado(TypedDict):
    pregunta: str
    informacion: str
    analisis: str
    revision: str
    informe: str
    iteraciones: int

def _prompt(rol, instruccion):
    return ChatPromptTemplate.from_messages([
        ("system", f"Eres el agente de {rol}. {instruccion}"),
        ("user", "{entrada}"),
    ])

# --- Nodo 1: RECOPILAR ---
def recopilar(estado: Estado) -> Estado:
    relevantes = [f for f in FUENTES if any(p in f.lower() for p in estado["pregunta"].lower().split())] or FUENTES
    info = "\n".join(f"- {f}" for f in relevantes)
    return {"informacion": info, "iteraciones": estado.get("iteraciones", 0) + 1}

# --- Nodo 2: ANALIZAR ---
def analizar(estado: Estado) -> Estado:
    cadena = _prompt("análisis", "Analiza la información, extrae ideas clave y relaciónalas con la pregunta.") | llm | StrOutputParser()
    return {"analisis": cadena.invoke({"entrada": f"PREGUNTA: {estado['pregunta']}\nINFORMACIÓN:\n{estado['informacion']}"})}

# --- Nodo 3: REVISAR ---
def revisar(estado: Estado) -> Estado:
    cadena = _prompt("revisión", "Detecta vacíos o inconsistencias. Responde SOLO 'OK' si es suficiente, o 'REPETIR' si falta información.") | llm | StrOutputParser()
    return {"revision": cadena.invoke({"entrada": f"PREGUNTA: {estado['pregunta']}\nANÁLISIS:\n{estado['analisis']}"}).strip().upper()}

def decidir(estado: Estado) -> Literal["recopilar", "sintetizar"]:
    if "REPETIR" in estado["revision"] and estado["iteraciones"] < 3:
        return "recopilar"
    return "sintetizar"

# --- Nodo 4: SINTETIZAR ---
def sintetizar(estado: Estado) -> Estado:
    cadena = _prompt("síntesis", "Redacta un informe final breve y fundamentado, citando solo la información disponible.") | llm | StrOutputParser()
    return {"informe": cadena.invoke({"entrada": f"PREGUNTA: {estado['pregunta']}\nANÁLISIS:\n{estado['analisis']}"})}

grafo = StateGraph(Estado)
grafo.add_node("recopilar", recopilar)
grafo.add_node("analizar", analizar)
grafo.add_node("revisar", revisar)
grafo.add_node("sintetizar", sintetizar)
grafo.add_edge(START, "recopilar")
grafo.add_edge("recopilar", "analizar")
grafo.add_edge("analizar", "revisar")
grafo.add_conditional_edges("revisar", decidir, {"recopilar": "recopilar", "sintetizar": "sintetizar"})
grafo.add_edge("sintetizar", END)

equipo = grafo.compile()
print("Equipo multiagente compilado.")

## 4. Ejecutar el equipo y leer el registro

In [ ]:
resultado = equipo.invoke({"pregunta": "¿Cómo se reduce la alucinación en los agentes?", "iteraciones": 0})

print("=== REGISTRO DE EJECUCIÓN ===")
print("Iteraciones realizadas:", resultado["iteraciones"])
print("\n--- Información recopilada ---")
print(resultado["informacion"])
print("\n--- Análisis ---")
print(resultado["analisis"])
print("\n--- Decisión de revisión ---")
print(resultado["revision"])
print("\n=== INFORME FINAL ===")
print(resultado["informe"])

## 5. CrewMaster: conceptos

[CrewMaster](https://pypi.org/project/crewmaster/) organiza equipos de agentes
mediante tres conceptos:

- **Operaciones:** unidades de trabajo (equivalentes a los nodos/roles anteriores).
- **Canales de colaboración:** cómo se comunican e intercambian resultados los agentes.
- **Evaluación:** criterios para juzgar la calidad de cada operación y decidir si repetir.

> ⚠️ **Nota técnica del curso:** CrewMaster **no** es CrewAI. Antes de elegir un
> modelo o driver, **verifica la compatibilidad de la versión instalada** con tu
> proveedor (p. ej. Ollama). No asumas que todos los drivers funcionan con Ollama.

En este entorno, `pip index versions crewmaster` **no encontró el paquete
publicado**; por eso el laboratorio implementa el flujo multiagente con
**LangGraph** (portable y verificable) y deja CrewMaster como referencia a
confirmar contra su documentación oficial.

Esqueleto conceptual (pseudocódigo; **verifica la API real** contra la versión instalada):

```python
# PSEUDOCÓDIGO - no ejecutar sin verificar la API de CrewMaster instalada.
# from crewmaster import Operation, Channel, Crew

# recopilar = Operation(name="recopilar", agent=busqueda)
# analizar   = Operation(name="analizar",   agent=analisis)
# revisar    = Operation(name="revisar",    agent=revision)
# sintetizar = Operation(name="sintetizar", agent=sintesis)
#
# canal = Channel(operations=[recopilar, analizar, revisar, sintetizar],
#                 evaluation=revisar)   # la evaluación decide si se repite una etapa
#
# equipo = Crew(channel=canal)
# resultado = equipo.run(pregunta="...")
```

## Resumen del laboratorio

- Diseñamos **4 roles** (recopilar, analizar, revisar, sintetizar).
- Coordinamos con **LangGraph**: la revisión puede **repetir** la recopilación.
- Registramos la **ejecución** y la **decisión de revisión**.
- Introdujimos **CrewMaster** y su nota de compatibilidad.

**Entregable del Módulo 6:** workflow multiagente con roles documentados,
intercambio de resultados, registro de ejecución y evaluación.